# SDS210 Final project - Wildfire Mapping in Borneo

This notebook builds an automated pipeline that fetches live satellite data from NASA FIRMS, cleans it and visualizes it on an interactive map.

**Research question:** Where are the most intense active fires in Borneo right now, and how is fire activity distributed across the region?

## 1. Imports

In [98]:
import pandas as pd  #data frames, read_csv, concat
import requests #api call
import folium
from folium.plugins import HeatMap
from io import StringIO

## 2. API Configuration

- NASA FIRMS created a free API for the near-real-time fire data (sent via emaail).
- The personal map key is needed and can be obtained here 
--> https://firms.modaps.eosdis.nasa.gov/api/
- the key goes into every data request

In [ ]:
#firms API

#still do in git ignore later

map_key   = "your_api_here"

#get url, hide map key 
url_api = 'https://firms.modaps.eosdis.nasa.gov/mapserver/mapkey_status/?map_key=' + map_key

#check status and how many transactions i have

response = requests.get(url_api)
print(response.status_code)
print(response.text[:200])


200
{ "transaction_limit" : 5000, "current_transactions": 0, "transaction_interval" : "10 minutes" }


## 3. Data aviability
- Quesry to see which sensors and date ranges are aviable through the API. 
- url will return info about all supported sensors and their corresponding datasets
- instead of 'all' i could also specify individual sensor like landsat_nrt
- The URL structure is always: 'base_url' + map_key + '/dataset_I_want'

In [100]:
##base_url / my_key / what_I_want
da_url = 'https://firms.modaps.eosdis.nasa.gov/api/data_availability/csv/' + map_key + '/all'
df = pd.read_csv(da_url)
display(df)

,data_id,min_date,max_date
0,MODIS_NRT,2026-03-01,2026-05-21
1,MODIS_SP,2000-11-01,2026-02-28
2,VIIRS_NOAA20_NRT,2026-04-01,2026-05-21
3,VIIRS_NOAA20_SP,2018-04-01,2026-03-31
4,VIIRS_NOAA21_NRT,2024-01-17,2026-05-21
5,VIIRS_SNPP_NRT,2026-04-01,2026-05-21
6,VIIRS_SNPP_SP,2012-01-20,2026-03-31
7,LANDSAT_NRT,2022-06-20,2026-05-21
8,GOES_NRT,2022-08-09,2026-05-21
9,BA_MODIS,2000-11-01,2026-02-01


## 4. Parameters

- The three VIIRS sensors are being chose, max coverage, near_real_time data
- define bouding box (borneo)
- time window


In [101]:
## Borneo BBOX: west, south, east, north

bbox = "108.5,-4.5,119.5,7.5"

# Sensoren: all three VIIRS NRT --> near real time
sensors = ["VIIRS_SNPP_NRT", "VIIRS_NOAA20_NRT", "VIIRS_NOAA21_NRT"]

# how many days i wanna look at
day_range = 1

## 5. Download and clean the fire data
The 'fetch()' function  builds the api_url for sensors and return dataframe. Calles once per sonsors, then all the results are stacked into one dataset

Quality filter remove low confidence detection 'l' and dead pixels with no measured 'frp=0'


In [102]:
def fetch(sensor):
    """
    Download near-real-time fire detections from the NASA FIRMS API
    for a single satellite sensor.

    Parameters
    ----------
    sensor : str
        FIRMS sensor identifier --> ex: 'VIIRS_SNPP_NRT'.

    Returns
    -------
    pandas.DataFrame
        DataFrame of fire detections with columns including latitude,
        longitude, frp, confidence, acq_date, and sensor name.

    Raises
    ------
    requests.HTTPError
        If the API request fails
    """
    url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{map_key}/{sensor}/{bbox}/{day_range}"
    r = requests.get(url, timeout=60)
    r.raise_for_status() #raises error if request failed
    df = pd.read_csv(StringIO(r.text))
    df["sensor"] = sensor #records which satellite detected this fire
    return df

# Download & combine all sensors and combine into one DataFrame
df = pd.concat([fetch(s) for s in sensors], ignore_index=True)

# Keep only nominal/high confidence detections with real fire energy
df = df[df["confidence"].isin(["n", "h"]) & (df["frp"] > 0)]

print(f"{len(df)} fire detections after cleaning")
print(df.groupby("sensor").size())
df.head()


212 fire detections after cleaning
sensor
VIIRS_NOAA20_NRT    55
VIIRS_NOAA21_NRT    76
VIIRS_SNPP_NRT      81
dtype: int64


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,sensor
0,2.92239,117.36320,341.98,0.6,0.70,2026-05-21,453,N,VIIRS,n,2.0NRT,288.17,7.37,D,VIIRS_SNPP_NRT
1,2.92310,117.36571,329.32,0.6,0.70,2026-05-21,453,N,VIIRS,n,2.0NRT,287.85,7.47,D,VIIRS_SNPP_NRT
2,3.13172,117.30460,347.98,0.6,0.70,2026-05-21,453,N,VIIRS,n,2.0NRT,288.96,9.56,D,VIIRS_SNPP_NRT
3,3.13197,117.30428,349.14,0.6,0.70,2026-05-21,453,N,VIIRS,n,2.0NRT,288.95,9.41,D,VIIRS_SNPP_NRT
4,3.22850,117.21288,326.75,0.6,0.71,2026-05-21,453,N,VIIRS,n,2.0NRT,288.67,5.55,D,VIIRS_SNPP_NRT


## 6. Create interactive map

Two layers combined to one folium map

1.  **Heatmap** --> all fire detections weighted by FRP. Shows the overall spatial pattern of fire activity across Borneo.

2. **Hot fire markers** — individual circle markers for the top 10% most intense fires (by FRP). By clicking the marker, one can see DRP vale, date and sensor. --> most energetically significant fires.

you can switch between the layer with the layer control 

In [103]:
def build_map(df, zoom_start=5):
    """
    Build  interactive Folium map of wildfire detections over Borneo.

    The map contains two layers:
    - A heatmap of all detections weighted by FRP intensity
    - Circle markers for the top 10% most intense fires

    Parameters
    ----------
    df : pandas.DataFrame
        Cleaned fire detections DataFrame containing at minimum:
        latitude, longitude, frp, acq_date, sensor columns.
    zoom_start : int, optional
        Initial zoom level for the map (default 5).

    Returns
    -------
    folium.Map
        Interactive Folium map with wildfire detections visualized.
        """


    m = folium.Map(location=[1.5, 114], zoom_start=zoom_start, tiles="CartoDB positron")


    # title, to the left of map
    title_html = ('<div style="position:fixed; top:10px; left:40px;' \
    'z-index:1000; background:white; padding:4px 8px;' \
    ' border-radius:4px; border:1px solid grey;' \
    ' font-size:10px; font-weight:bold;">Active Wildfires in Borneo, last 24h</div>')

    m.get_root().html.add_child(folium.Element(title_html))

    # Layer 1: Heatmap layer (weighted by FRP)
    heat_data = df[["latitude", "longitude", "frp"]].values.tolist()
    HeatMap(heat_data, radius=8, blur=6, name="FRP heatmap").add_to(m)

    # Layer 2: dot markers for high-FRP fires (top 10%)
    threshold = df["frp"].quantile(0.90)
    hot = df[df["frp"] >= threshold]

    dots = folium.FeatureGroup(name=f"Top 10% fires")
    for _, row in hot.iterrows():
        folium.CircleMarker(
            location=[row["latitude"], row["longitude"]],
            radius=4,
            color="darkred",
            fill=True,
            fill_opacity=0.7,
            popup=folium.Popup(
                f"<b>FRP:</b> {row['frp']:.1f} MW<br>"
                f"<b>Date:</b> {row['acq_date']}<br>"
                f"<b>Sensor:</b> {row['sensor']}",
                max_width=200
            ),  
        ).add_to(dots)
    dots.add_to(m)

# legend
    legend_html = f"""
    <div style="position:fixed; bottom:10px; left:10px; z-index:1000; 
        background:white; padding:5px 8px; border-radius:4px; 
        border:1px solid grey; font-size:9px; line-height:1.4;">
        <b>FRP intensity</b><br>
        <span style="background:linear-gradient(to right, blue, green, yellow, red); 
            display:inline-block; width:70px; height:7px; border-radius:2px;"></span><br>
        <span>Low</span><span style="float:right">High</span><br>
        ● top 10% (≥ {threshold:.0f} MW)
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend_html))

# smaller layer control
    folium.LayerControl(collapsed=True).add_to(m)

    return m

# Save
m = build_map(df)
m.save("../outputs/borneo_fires.html")
print("Saved → outputs/borneo_fires.html")
m

Saved → outputs/borneo_fires.html


## 7. Summary

This pipeline automatically downloads, cleans, and maps near-real-time  wildfire data for Borneo from three VIIRS satellite sensors.

What are the findings: 
- Heatmap shows regional fire hotspots
- top 10% most intense fires (by FRP) are mapped individually as high risk
-Popup informations allow to get more information on intense fire events

In [ ]:
fig